In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw9.ipynb")

# Homework 9: Studying Employee Attrition With K-Means

### Collaborators and Sources
In addition to recording your **collaborators** (TAs, peers, group members, roommates, etc.) on this homework in the cell below, you are required to **cite/indicate all external sources** used when finishing this assignment. 
External sources are defined as anything that is not considered course material, such as online sources (webpages, blog posts, etc.), content generated by AI systems, books, etc.

When using external sources, indicate what kind of external sources (e.g. stack overflow, WashU chatGPT, etc.) you used in the cell below and then provide more specific citations (such as links to webpages or in case of AI generated asnwers the actual sources (links provided by the AI system) and the prompt used to generate the answers) with your answer to **each specific problem**.  


Note that these citations will not free you from your obligation to submit your _own_ code and write-ups, however, they will be taken into account during the grading and regrading process, **especially** when two or more submissions closely resemble each other. Working with each other is ok, as long as you cite who you worked with and you don't copy anyone's answers directly!

### Submission instructions
* Submit this Python notebook on Gradescope.
* Execute all cells and save your notebook prior to submission. 
* Don't forget to commit and push to your Github repo!
* **Do not change the number of cells!** Your submission notebook should have exactly one code cell per problem. 
* Do **not** remove the `# your code here` line. Add your solution **after** that line.

### Complete this week's Lab first!
This hw assignment is a continuation of this week's lab. We expect you to **review and complete** the lab prior to doing this hw assignemnt. If you did not get to complete the lab in-class, we expect you to **finish it now**. Almost all parts and specifically the ones towards the end are **highly relevant for and connected to this assignment**. _Parts labeled **Optional** are for the interested student._ 

## Let's get started

In this homework, we will be exploring a more realistic application of clustering. Review and complete **Lab 9 (K-Means Clustering)** first. Most of the things we ask you to do in this homework are explained in the lab. In general, you should feel free to import any package that we have previously used in class. Ensure that all plots have the necessary components that a plot should have (e.g. axes labels, a title, a legend).

### Learning Goals
* **[G9-2]** Understand the **k-means algorithm**. 
* **[G9-5]** Implement the **k-means algorithm** by using `sklearn`. 
* **[G9-6]** Visually **evaluate k-means clustering results** for 2d data.
* **[G9-7]** Know how to use the **elbow method** to select k.

#### Practice Python skills 
* plotting with `matplotlib`
* use `sklearn` for ML

### Some imports and configurations

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1" # should stop kernel from dying when fitting a k-means model

import numpy as np
import matplotlib.pyplot as plt

from utility.util_hw import load_toy, configure_plots
from utility.util_hw import sample_centroids, fit, plot_kmeans

configure_plots()

## 1. Do the Initial Centroids Matter?

Let's investigate if the $k$-means algorithm is sensitive to the initial starting points. In the cell below, we generate a toy dataset with **five** clusters.

In [ ]:
X, _ = load_toy(500, 5, width=0.07, random_state=4)

<!-- BEGIN QUESTION -->

### Problem 1.1

Let's take a quick peek at what the data looks like.

**Do this!** Plot the toy data $X$. Make sure that your plot has all of the necessary components.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], s=20)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Toy Data X")
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Problem 1.2

**Do this!** Using the functions `sample_centroids`, `fit`, and `plot_kmeans`, experiment with different `random_state`s to see if you can observe _different_ final centroids depending on the initial starting points. Use the data sampled above and produce two plots in the two code cells provided below; The first should show a reasonable $k$-means solution for k=5, whereas the second one should show a "wrong" solution (again for k=5). (Where "wrong" means visually less correct.)
> **Hint:** `fit` as imported from `util_hw` returns two arrays, the centroids **and** assignments.  

In [ ]:
init_centroids = sample_centroids(X, 5, random_state=1)
centroids, assignments = fit(X, 5, initial=init_centroids)
plot_kmeans(X, centroids, init_centroids, assignments)
plt.title("Reasonable k-means solution (k=5)")
plt.show()

In [ ]:
init_centroids = sample_centroids(X, 5, random_state=12)
centroids, assignments = fit(X, 5, initial=init_centroids)
plot_kmeans(X, centroids, init_centroids, assignments)
plt.title("Less correct k-means solution (k=5)")
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Problem 1.3

**Write-up!** You should have seen quite different clusterings based on different inital centroids. Carefully explain why this happens by thinking about the clustering plroblem and how k-means solves it, as well as, what in the algorithm is causing this to happen? 
Then suggest an improvement to the part in our k-means algorithm that causes this issue. 

K-means delivers different outcomes because the first centroids steer the search. The procedure never guarantees the overall best partition. It begins with the selected centroids then keeps sending each record to the closest centroid and moves every centroid to the average of its current members. Distinct starting positions often end in distinct final groupings.

The core trouble lies in the random pick of the first centroids. A poor first layout lets the procedure settle on a merely local, sub optimal grouping.

A practical fix is to switch to k-means++ for the first centroids instead of a pure random choice. A further safeguard is to repeat k-means from many random starts and retain the partition with the lowest within cluster sum of squares.

<!-- END QUESTION -->

## 2. Using `sklearn` for $k$-means 

In this section, we will explore the [$k$-means model from `sklearn`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) and discuss some of the additional features supplied by their implementation. Before we begin, it is suggested that you work through Lab 7 if you haven't already as we will assume familiarity with the terms used there.

### Looking Into the Model

Now let's create a new $k$-means model and learn about it's interface. In general, you will find that the $k$-means model from `sklearn` shares a lot of the same methods as the other models that we have looked at. However, there are some differences that are notable.

In [ ]:
from sklearn.cluster import KMeans

model = KMeans()

Use the IPython `?` operator to answer **all of the problems** below.

In [ ]:
# use this cell to explore the interface of the KMeans model --> Do not add more cells below.
KMeans?

### Problem 2.1

**Write-up!** How do you specify the number of clusters you would like to fit? Specify the name of the argument as a string. 

> **Hint:** If it was the random state argument your solution below would be `my_string = 'random_state'`.

*Note: this question has hidden tests, so follow the above instructions carefully.*


The argument is **`n_clusters`**. It is used to specify how many clusters the KMeans model should fit.

In [ ]:
my_string = 'n_clusters'

<!-- BEGIN QUESTION -->

### Problem 2.2

**Write-up!** How does the algorithm initialize the centroids by default and why is this initialization better than randomly choosing starting centroids? How does the algorithm decide that the centroids have converged? 

sklearn starts k-means with the k-means++ rule. k-means++ places the first centre far from the previous ones - the initial set is well scattered. A scattered start lowers the risk of poor seeds and normally gives steadier, higher quality clusters.

Iteration halts when the centres shift less than the tolerance value (`tol`).

<!-- END QUESTION -->

## 3. Clustering the Toy Data with `sklearn`

Let's try using `sklearn` to cluster our data.

### Problem 3.1

**Do this!** Create and fit a _new_ `KMeans` model of our data with **the default arguments** except for `random_state` which should be set to 11. Be sure to store the fit centroids in `centroids` and cluster assignment indicees in `assignments` for later use.
> **Hint:** Check the scikit-learn [`kMeans` 🔗](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) documentation for how to get the centroids and assignments or use `KMeans?` and `KMeans.` + `tab`.

In [ ]:
toy_model = KMeans(random_state=11)
toy_model.fit(X)

toy_assignments = toy_model.labels_
toy_centroids = toy_model.cluster_centers_

In [ ]:
grader.check("q3ai")

<!-- BEGIN QUESTION -->

**Do this!** How many centroids were fitted by the model? How many points were assigned to each cluster? 

This should be done in **one `numpy` function** that gets both the list of cluster ids _and_ the number of points assigned to each cluster. 

In [ ]:
np.unique(toy_assignments, return_counts=True)

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Problem 3.2

**Do this!** Create a plot showing the centroids that were produced by the model and the data points colored by their cluster assignment. Be sure to include all necessary plot components and remember that presentation matters.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(X[:, 0], X[:, 1], c=toy_assignments, s=30, alpha=0.8, label="data points")
plt.scatter(toy_centroids[:, 0], toy_centroids[:, 1], s=200, c="red", marker="X", label="centroids")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("KMeans clustering results")
plt.legend()
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Write-up!** In that case that we have 2d data (as fo rour toy example) we can visually evalaute the quality of the k-means clustering. Using your visualization above, evaluate the result of our k-means algorithm. Provide a detailed evaluation describing the clustering result. Go beyond simply stating that this is a good or bad clustering. Discuss why the clustering turned out good/bad. 

The outcome lacks quality - the algorithm produced 8 clusters but the data form about 5 coherent bundles. With the excess number, authentic bundles on the right were fractured into tinier fragments.

The left and central bundles were grouped more successfully because their boundaries stand out. Broadly speaking, the partition is only partially correct and the core issue is that the chosen k exceeds the true count.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Deciding How Many Clusters to Use
As we have seen for $k$-means, $k$ is the number of clusters/centroids that the algorithm will try to find. Choosing $k$ is an important task as it determines the output of the algorithm. Since $k$ is a parameter to our clustering algorithm, we can use a **model selection** strategy to do this.


### Problem 3.3

Consider the Sum of Squared Distances $SSD_j$ as the sum of all points in the $j$th cluster to its corresponsing cluster center $c_j$: 
$$SSD_j = \sum_{C(i)=j} \sum_{C(i')=j}  \;d(x_i,x_{i'})^2,$$
where $C(i)=j$ means that $x_i$ belongs to cluster $j$. 

Then, the objective function that $k$-means optimizes is the sum of the $SSD_j$ over all clusters $WCSS = \sum_j SSD_j$ (_Within Cluster Sum of Squares_). This means that, we want to find clusters of points that are close to one another. We can estimate how close the cluster points are to one another by measuring how far each point assigned to the cluster is from its center.

**Write-up!** Why is it difficult to find the right number of clusters for a general clustering task with d-dimensional input? Think about the value of the objective function $WCSS$ we are optimizing with respect to the number of clusters $k$.

Selecting the correct count of clusters presents a challenge because the objective function WCSS tends to fall as the parameter k grows. Each time we raise k, the algorithm places points into narrower and more compact groups - the gap between each point and its group centre shrinks. At the limit, when each point forms a separate cluster, WCSS reaches zero.

For that reason, the lowest possible WCSS by itself does not reveal the optimal k. A higher k almost always scores better on the objective function, even when it splits the data too finely plus fails to reflect the real pattern.

<!-- END QUESTION -->

In the cell below, we will generate a dataset with an _unknown_ number of clusters.

In [ ]:
X_unknown, _ = load_toy(1000, k=-1, random_state=6)

<!-- BEGIN QUESTION -->

**Do this!** Use the elbow method described in the lecture to find a good clustering for our data. Produce a plot that shows the model performance $SSD^{(k)}$ as a function of $k \in [1, 10]$, where $SSD^{(k)} = \sum_{j=1}^k SSD_j$.  Make sure to create new models when appropriate.

In [ ]:
from utility.util_hw import squared_distance

ks = np.arange(1, 11)
ssd_values = []

for k in ks:
    model = KMeans(n_clusters=k, random_state=11)
    model.fit(X_unknown)
    
    assignments = model.labels_
    centroids = model.cluster_centers_
    
    ssd_k = 0
    for i in range(X_unknown.shape[0]):
        ssd_k += squared_distance(X_unknown[i], centroids[assignments[i]])
    
    ssd_values.append(ssd_k)

plt.figure(figsize=(7, 5))
plt.plot(ks, ssd_values, marker='o')
plt.xlabel("k")
plt.ylabel("SSD^(k)")
plt.title("Elbow Plot for KMeans")
plt.xticks(ks)
plt.show()

<!-- END QUESTION -->

**Write-up!** Choose an appropriate $k$ using your ellbow plot.

> **Hint:** If your answer would be 1000 then your solution below should be `my_k_toy = 1000`.

*Note: this question has hidden tests, so follow the above instructions carefully.*


The elbow appears at (k=5), so this is a reasonable choice for the number of clusters.


In [ ]:
my_k_toy = 5

**Do this!** In the following cell, plot the dataset to see if your choice is reasonable. 
> **Note:** This kind of visual verification/evaluation is _only_ possible for 2D input data. 

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X_unknown[:, 0], X_unknown[:, 1], s=20)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Unknown Toy Dataset")
plt.show()

In [ ]:
grader.check("q3ciii")

## import pandas as pd
data = pd.read_csv('./utility/HR-Employee-Attrition.csv')4. Tackling Employee Attrition


A real problem that Human Resources (HR) departments in companies all over the world would like to address is employee attrition, or turnover. They would like to reduce the number of employees who leave the company as hiring new employees is expensive. In this section, we would like to see if we can make use of $k$-means to identify patterns in employee attrition so that we might suggest which areas an HR department should focus on.

To show off their Watson platform, IBM released a (fictional) sample dataset in 2015 containing employee statistics and whether or not they left the company. We'll use this dataset in our own exercise.

In [ ]:
import pandas as pd
data = pd.read_csv('./utility/data/HR-Employee-Attrition.csv')

With our problem in mind, the next thing to do is to explore and then process our data. Use the cell below for data exploration, i.e, to do the next problem. 

In [ ]:
# use this cell to explore! --> Do not add more cells below.
data.head()
data.shape
data.columns
data.info()
data.describe(include='all')

<!-- BEGIN QUESTION -->

### Problem 4.1

**Write-up!** Describe the data in `HR-Employee-Attrition.csv` (`data`), answering the following questions: 
* How many examples and features does the dataset have?
* What types of features are in the dataset and how many features are of each type?
* Look at all the features and their stats and then name two features for each type and list which values these features can take.
* Last, **ask one question** or **pose one hypothesis** you are interested in invesitgating with this data (stick with somehting that is doable/possible to investigate with the available data).
> **Hint:** Consider the steps of EDA; what would you like to know about this dataset?

The dataset has 1,470 examples and 35 features. There are two main feature types in the dataset: 26 numeric featuresand 9 categorical features.

For the categorical features, two examples are Attrition and BusinessTravel. Attrition can take the values Yes or No. BusinessTravel can take Non-Travel, Travel_Rarely, or Travel_Frequently. For the numeric features, two examples are Education and JobSatisfaction. Education takes the values 1, 2, 3, 4, 5, and JobSatisfaction takes the values 1, 2, 3, 4.

One question I would like to investigate is: Are employees who work overtime more likely to leave the company? This seems possible to study with the available data because the dataset includes both OverTime and Attrition.

<!-- END QUESTION -->

### Problem 4.2

Now that we have a feel for what our data looks like, let's do some _data wrangling_.

**Do this!** Use the cell below for the steps that you need to prepare this data for further analysis. Make sure that your analysis and work is presented well and effectively communicates your work. In this process, consider whether each feature is informative (eg. EmployeeNumber might not be) and remove those that are not from your dataset. Do not (necessarily) remove features with missing values!
> **Hint:** This is somewhat subjective, but there are more then one column that aren't necessary.
> 
> **Hint:** You can use the `pandas.DataFrame.drop` function. Make sure the data after processing is still stored in `data`. 

*Note: this question has hidden tests, so follow the above instructions carefully.*

In [ ]:
data = data.drop(columns=['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours'], errors='ignore')

**Write-up!** Describe and explain the data wrangling steps below. Which features did you remove? Why did you select these features to drop?

The report states the same action in words
I excluded EmployeeNumber, EmployeeCount, Over18 & StandardHours. 
EmployeeNumber is only an identifier - it carries no signal about who leaves the company. EmployeeCount, Over18 & StandardHours hold identical entries across all rows - they supply no contrast and add no insight. 

My overall cleaning task consisted of stripping away every field that gives no analytical value. All remaining variables stay in the table because each one still has potential to expose reasons for staff departure.

In [ ]:
grader.check("q4b")

### Problem 4.3

Another step to do in our data processing phase is to replace categorical variables that are represented as strings with an enumeration. For example, `'Attrition'` has `'Yes'` and `'No'` values that we would like to encode as `1` and `0` respectively.

**Do this!** In the following cell, [`replace`](http://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html) the string values in categorical variables with enumerations. Make uses of the `encoded` `DataFrame` which is a copy of `data`. 
> **Hint:** You can use the `unique` and `enumerate` functions to help you do this.

In [ ]:
encoded = data.copy()
pd.set_option('future.no_silent_downcasting', True)

for col in encoded.columns:
    if encoded[col].dtype == 'object':
        if col == 'Attrition':
            encoded[col] = encoded[col].replace({'No': 0, 'Yes': 1}).astype('int64')
        else:
            values = encoded[col].unique()
            mapping = {value: i for i, value in enumerate(values)}
            encoded[col] = encoded[col].replace(mapping).astype('int64')
    else:
        encoded[col] = encoded[col].astype('int64')

In [ ]:
grader.check("q4c")

<!-- BEGIN QUESTION -->

### Problem 4.4

Now that we have a processed dataset, let's move on to forming clusters with $k$-means. Normally, we would do some EDA here, but in the interest of time, we will forgo that part of the data science workflow. If you want to, we still encourage you to do so.

That being said, we will need to prepare an $X$ matrix of our dataset. At this point, we will drop the `'Attrition'` column from our dataset. We will also [scale our data](https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling). 

In [ ]:
from sklearn.preprocessing import scale

X_processed = scale(np.float64(encoded.copy().drop('Attrition', axis=1)))

**Do this!** In the cell below, build an elbow plot for $k \in [1, 20]$ as you did in [Problem 3.3](#Problem-3.3).

In [ ]:
ks = np.arange(1, 21)
ssd_values = []

for k in ks:
    model = KMeans(n_clusters=k, random_state=11)
    model.fit(X_processed)
    
    assignments = model.labels_
    centroids = model.cluster_centers_
    
    ssd_k = 0
    for i in range(X_processed.shape[0]):
        ssd_k += squared_distance(X_processed[i], centroids[assignments[i]])
    
    ssd_values.append(ssd_k)

plt.figure(figsize=(7, 5))
plt.plot(ks, ssd_values, marker='o')
plt.xlabel("k")
plt.ylabel("SSD^(k)")
plt.title("Elbow Plot for Employee Attrition Data")
plt.xticks(ks)
plt.show()

<!-- END QUESTION -->

Often, the elbow plot may not have a clear elbow. We can gather additional evidence using the Silhouette Coefficient, where a higher Silhouette Coefficient score relates to a model with better defined clusters. The Silhouette Coefficient is defined for each sample and is composed of two scores:

`a`: The mean distance between a sample and all other points in the same cluster.

`b`: The mean distance between a sample and all other points in the next nearest cluster.

The Silhouette Coefficient `s` for a single sample is then given as:

$$s = \frac{b-a}{\max\{a,b\}}$$

The Silhouette Coefficient for a set of samples is given as the mean of the Silhouette Coefficient for each sample.

<!-- BEGIN QUESTION -->

**Do this!** In the cell below, build a bar chart of the Silhouette Coefficients for $k \in [2, 20]$.

In [ ]:
from sklearn.metrics import silhouette_score

ks = np.arange(2, 21)
sil_scores = []

for k in ks:
    model = KMeans(n_clusters=k, random_state=11)
    labels = model.fit_predict(X_processed)
    score = silhouette_score(X_processed, labels)
    sil_scores.append(score)

plt.figure(figsize=(8, 5))
plt.bar(ks, sil_scores)
plt.xlabel("k")
plt.ylabel("Silhouette Coefficient")
plt.title("Silhouette Scores for Employee Attrition Data")
plt.xticks(ks)
plt.show()

<!-- END QUESTION -->

**Do this!** Using the information from both the elbow plot and the silhouette coefficients, choose a 𝑘 and assign it to `my_k`.

*Note: this question has hidden tests, so follow the above instructions carefully.*

In [ ]:
my_k = 5

**Write-up!** Brielfy explain why you chose this 𝑘.

I chose k = 5 because the elbow plot starts to flatten around 5, so adding more clusters gives smaller improvement. Using the silhouette scores together with the elbow plot, 5 seems like a reasonable balance between model fit and cluster quality.

In [ ]:
grader.check("q4diii")

### Problem 4.5

**Do this!** Using the $k$ you selected in [Problem 4.4](#Problem-4.4), create and fit a new model. Remember to save the cluster assignments and centroids. Use a random state of 11.

In [ ]:
k = my_k
new_model = KMeans(n_clusters=k, random_state=11)
new_model.fit(X_processed)

new_assignments = new_model.labels_
new_centroids = new_model.cluster_centers_

In [ ]:
grader.check("q4e")

### Problem 4.6

Now that we have cluster assignments from $k$-means, we need to analyze the significance of each cluster. To do that, let's return to our original DataFrame, `data`.

In the following cell, we add our cluster assignments to `data_aug`. We also compute a pivot table which provides a summary of each cluster with the within cluster `mean` for each feature.

In [ ]:
data_aug = data.copy()
data_aug['Cluster'] = new_assignments
pivot = data_aug.select_dtypes(include='number').groupby(data_aug['Cluster']).mean()
pivot

**Do this!** Compute the percentage of total attrition accounted for by each cluster and store the result in `pivot['% of Attrition']`.
> **Hint:** The dataframe `encoded` still has attrition included. Use it!

In [ ]:
encoded_aug = encoded.copy()
encoded_aug['Cluster'] = new_assignments

attrition_by_cluster = encoded_aug.groupby('Cluster')['Attrition'].sum()
total_attrition = encoded_aug['Attrition'].sum()
pivot['% of Attrition'] = 100 * attrition_by_cluster / total_attrition

In [ ]:
grader.check("q4f")

<!-- BEGIN QUESTION -->

### Problem 4.7

Let's take a look at the results and identify potential areas for intervention to suggest to the HR department.

In [ ]:
pivot.sort_values(by='% of Attrition', axis=0).T

**Write-up!** Describe the clusters produced and interpret their meaning. What makes each one a separate cluster? Is there anything that stands out with respect to attrition rate? What might you suggest HR look into to improve employee retention?

The clusters mainly differ by **age, income, job level, and experience**. For example, **Cluster 1** has the oldest and most experienced employees, with the highest job level and monthly income, and it also has the **lowest attrition percentage**. In contrast, **Cluster 0** has the youngest employees, the lowest job level, the lowest income, and the fewest total working years, and it has the **highest attrition percentage**.

This suggests that younger and less established employees may be more likely to leave, while older and higher-level employees are more likely to stay. Cluster 2 also has a relatively high attrition percentage, so it may represent another group that is at higher risk of leaving.

HR should pay special attention to the high-attrition clusters, especially younger and lower-income employees. Possible actions include improving early-career support, promotion opportunities, compensation, and retention programs for newer employees.


<!-- END QUESTION -->

**That's it! Follow the submission instructions below:**
* Restart your kernel and run all the cells
* Save your notebook
* Upload the .ipynb notebook directly to Gradescope